In [1]:
import sys
from pathlib import Path

import pandas as pd

try:
    import seaborn as sns
    HAS_SEABORN = True
    sns.set_theme(style="whitegrid")
except ImportError:
    HAS_SEABORN = False

sys.path.append(str(Path("..").resolve() / "src"))

from config.config import setup

cfg = setup()

HAR_ROOT = cfg["HAR_ROOT"]

In [2]:
def _fix_duplicate_feature_names(names):
    seen, out = {}, []
    for n in names:
        if n in seen:
            seen[n] += 1
            out.append(f"{n}__{seen[n]}")
        else:
            seen[n] = 0
            out.append(n)
    return out


def load_har_data():
    har_root: Path | None = None

    if har_root is None:
        har_root = HAR_ROOT
    features = pd.read_csv(har_root / "features.txt",
                           sep=r"\s+", header=None, names=["idx", "name"])
    feature_names = _fix_duplicate_feature_names(features["name"].tolist())

    activities = pd.read_csv(har_root / "activity_labels.txt",
                             sep=r"\s+", header=None, names=["id", "label"])
    activity_map = dict(zip(activities["id"], activities["label"]))

    X_train = pd.read_csv(har_root / "train" / "X_train.txt",
                          sep=r"\s+", header=None, names=feature_names)
    y_train = pd.read_csv(har_root / "train" / "y_train.txt",
                          sep=r"\s+", header=None, names=["activity_id"])["activity_id"]
    y_train = y_train.map(activity_map).astype("category")
    groups_train = pd.read_csv(har_root / "train" / "subject_train.txt",
                               sep=r"\s+", header=None, names=["subject"])["subject"]

    X_test = pd.read_csv(har_root / "test" / "X_test.txt",
                         sep=r"\s+", header=None, names=feature_names)
    y_test = pd.read_csv(har_root / "test" / "y_test.txt",
                         sep=r"\s+", header=None, names=["activity_id"])["activity_id"]
    y_test = y_test.map(activity_map).astype("category")
    groups_test = pd.read_csv(har_root / "test" / "subject_test.txt",
                              sep=r"\s+", header=None, names=["subject"])["subject"]

    return X_train, y_train, groups_train, X_test, y_test, groups_test, feature_names


X_train, y_train, groups_train, X_test, y_test, groups_test, feature_names = load_har_data()

print(f"X_train: {X_train.shape} | y_train: {y_train.shape} | osoby w train: {groups_train.nunique()}")
print(f"X_test : {X_test.shape}  | y_test : {y_test.shape}  | osoby w test : {groups_test.nunique()}")
print(f"Liczba cech: {len(feature_names)}")
print(f"Klasy: {list(y_train.cat.categories)}")

X_train: (7352, 561) | y_train: (7352,) | osoby w train: 21
X_test : (2947, 561)  | y_test : (2947,)  | osoby w test : 9
Liczba cech: 561
Klasy: ['LAYING', 'SITTING', 'STANDING', 'WALKING', 'WALKING_DOWNSTAIRS', 'WALKING_UPSTAIRS']


In [4]:
assert X_train.shape[1] == 561, "Powinno być 561 cech."
assert X_train.shape[0] == y_train.shape[0] == groups_train.shape[0]
assert X_test.shape[0]  == y_test.shape[0]  == groups_test.shape[0]

print(f"NaN w X_train: {int(X_train.isna().sum().sum())}")
print(f"NaN w X_test : {int(X_test.isna().sum().sum())}")
print(f"\nZakres X_train: [{X_train.values.min():.4f}, {X_train.values.max():.4f}]  (oczekiwane: [-1, 1])")
print(f"Zakres X_test : [{X_test.values.min():.4f}, {X_test.values.max():.4f}]")

train_subjects = set(groups_train.unique())
test_subjects  = set(groups_test.unique())
print(f"\nOsoby w train ({len(train_subjects)}): {sorted(train_subjects)}")
print(f"Osoby w test  ({len(test_subjects)}): {sorted(test_subjects)}")
print(f"Część wspólna (powinna być pusta): {train_subjects & test_subjects}")

NaN w X_train: 0
NaN w X_test : 0

Zakres X_train: [-1.0000, 1.0000]  (oczekiwane: [-1, 1])
Zakres X_test : [-1.0000, 1.0000]

Osoby w train (21): [np.int64(1), np.int64(3), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(11), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(19), np.int64(21), np.int64(22), np.int64(23), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30)]
Osoby w test  (9): [np.int64(2), np.int64(4), np.int64(9), np.int64(10), np.int64(12), np.int64(13), np.int64(18), np.int64(20), np.int64(24)]
Część wspólna (powinna być pusta): set()
